# HW03 — Richard Kwan Assignment 03, Problem 4

### 1 setup.py

In [9]:
# %load setup.py
"""Step 1: Import the libraries and select the compute device."""

import torch
from torch import nn
import torchvision
import torchmetrics
import matplotlib.pyplot as plt

# Prefer an NVIDIA GPU, then an Apple GPU, then the CPU.
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

if __name__ == "__main__":
    print(f"Using device: {device}")


Using device: cpu


### 2 load_data.py

In [10]:
# %load load_data.py
"""Load FashionMNIST and create training, validation, and test loaders."""

from pathlib import Path

import torch
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import FashionMNIST
from torchvision.transforms import v2


def load_data(root=Path(__file__).resolve().parent / "data"):
    """Return batch-size-32 loaders with a reproducible 55,000/5,000 split."""
    transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ])
    training_data = FashionMNIST(
        root=root, train=True, download=True, transform=transform
    )
    test_data = FashionMNIST(
        root=root, train=False, download=True, transform=transform
    )
    train_data, valid_data = random_split(
        training_data,
        [55_000, 5_000],
        generator=torch.Generator().manual_seed(42),
    )

    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    valid_loader = DataLoader(valid_data, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
    return train_loader, valid_loader, test_loader


if __name__ == "__main__":
    train_loader, valid_loader, test_loader = load_data()
    print(f"Training samples: {len(train_loader.dataset):,}")
    print(f"Validation samples: {len(valid_loader.dataset):,}")
    print(f"Test samples: {len(test_loader.dataset):,}")


NameError: name '__file__' is not defined

### 3 inspect_data.py

In [ ]:
# %load inspect_data.py
"""Print a training sample's details and display labeled FashionMNIST images."""

import matplotlib.pyplot as plt

from load_data import load_data


def main():
    train_loader, _, _ = load_data()
    train_data = train_loader.dataset
    # random_split returns a Subset; class names belong to its source dataset.
    class_names = train_data.dataset.classes

    image, label = train_data[0]
    print(f"Shape: {tuple(image.shape)}")
    print(f"Dtype: {image.dtype}")
    print(f"Class name: {class_names[label]}")

    fig, axes = plt.subplots(2, 3, figsize=(9, 6))
    for index, ax in enumerate(axes.flat):
        image, label = train_data[index]
        ax.imshow(image.squeeze(0).numpy(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(class_names[label])
        ax.axis("off")

    fig.suptitle("FashionMNIST training samples")
    fig.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()


### 4 model.py

In [ ]:
# %load model.py
"""Step 4: define the ImageClassifier MLP and the loss function.

Later steps import from this module, e.g.
`from model import ImageClassifier, model, xentropy`.
"""

import torch
import torch.nn as nn

from setup import device


class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )

    def forward(self, X):
        return self.mlp(X)


torch.manual_seed(42)
model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=300, n_hidden2=100,
                        n_classes=10).to(device)
xentropy = nn.CrossEntropyLoss()

if __name__ == "__main__":
    print(model)
    n_params = sum(param.numel() for param in model.parameters())
    print(f"parameters: {n_params:,}")
    X_dummy = torch.rand(32, 1, 28, 28, device=device)
    print(f"output shape for a batch of 32: {tuple(model(X_dummy).shape)}")


### 5 train_utils.py

In [ ]:
# %load train_utils.py
"""Training and TorchMetrics evaluation helpers adapted from the notebook."""

import torch


def evaluate_tm(model, data_loader, metric):
    """Evaluate a TorchMetrics metric over a complete data loader."""
    device = next(model.parameters()).device
    metric = metric.to(device)
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
           n_epochs):
    """Train and return per-epoch loss, training metric, and validation metric.

    Pass a TorchMetrics multiclass accuracy metric with num_classes=10 for
    FashionMNIST. History keys match the notebook: train_losses,
    train_metrics (training accuracy), and valid_metrics (validation accuracy).
    Loss is averaged over batches, as in the notebook.
    """
    device = next(model.parameters()).device
    metric = metric.to(device)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        metric.reset()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                metric.update(y_pred.detach(), y_batch)

        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item()
        )
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history


### 6 train.py

In [ ]:
# %load train.py
"""Train FashionMNIST for 20 epochs and save the history and model weights."""

import json
from pathlib import Path

import torch
from torchmetrics import Accuracy

from load_data import load_data
from model import ImageClassifier, loss_fn
from setup import device
from train_utils import train2


def main():
    torch.manual_seed(42)
    print(f"Using device: {device}")
    train_loader, valid_loader, _ = load_data()
    model = ImageClassifier().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    accuracy = Accuracy(task="multiclass", num_classes=10, average="micro").to(device)

    history = train2(
        model, optimizer, loss_fn, accuracy, train_loader, valid_loader,
        n_epochs=20,
    )

    output_dir = Path(__file__).resolve().parent / "artifacts"
    output_dir.mkdir(parents=True, exist_ok=True)
    history_path = output_dir / "history.json"
    history_path.write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")

    # Save CPU weights so the model can also be loaded without a GPU.
    model_path = output_dir / "model.pt"
    torch.save(
        {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()},
        model_path,
    )
    print(f"Saved training history to {history_path}")
    print(f"Saved model weights to {model_path}")


if __name__ == "__main__":
    main()


### 7 plot_accuracy.py

In [ ]:
# %load plot_accuracy.py
"""Plot saved training and validation accuracy against epoch."""

import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def main():
    project_dir = Path(__file__).resolve().parent
    history_path = project_dir / "artifacts" / "history.json"
    if not history_path.is_file():
        raise SystemExit("Training history is missing. Run python train.py first.")

    history = json.loads(history_path.read_text(encoding="utf-8"))
    train_accuracy = history["train_metrics"]
    valid_accuracy = history["valid_metrics"]
    if not train_accuracy or len(train_accuracy) != len(valid_accuracy):
        raise ValueError("Training and validation accuracy must have equal, nonzero lengths.")
    epochs = range(1, len(train_accuracy) + 1)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, train_accuracy, "o-", label="Training accuracy")
    ax.plot(epochs, valid_accuracy, "s-", label="Validation accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("FashionMNIST training and validation accuracy")
    ax.set_ylim(0, 1)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    output_path = project_dir / "training_accuracy.png"
    fig.savefig(output_path, dpi=150)
    plt.close(fig)
    print(f"Saved accuracy plot to {output_path}")


if __name__ == "__main__":
    main()


### 8 predict.py

In [ ]:
# %load predict.py
"""Display predictions and probabilities for the first three validation images."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from load_data import load_data
from model import ImageClassifier


def main():
    model_path = Path(__file__).resolve().parent / "artifacts" / "model.pt"
    if not model_path.is_file():
        raise SystemExit("Trained weights are missing. Run python train.py first.")

    model = ImageClassifier()
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
    model.eval()

    _, valid_loader, _ = load_data()
    valid_data = valid_loader.dataset
    class_names = valid_data.dataset.classes
    samples = [valid_data[index] for index in range(3)]
    images = torch.stack([image for image, _ in samples])

    with torch.inference_mode():
        probabilities = torch.softmax(model(images), dim=1)
        predictions = probabilities.argmax(dim=1)
        top_probs, top_classes = probabilities.topk(4, dim=1)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for index, (ax, (image, true_label)) in enumerate(zip(axes, samples)):
        predicted_name = class_names[predictions[index].item()]
        true_name = class_names[true_label]
        print(f"\nValidation image {index + 1}")
        print(f"Predicted: {predicted_name} | True: {true_name}")
        print("Softmax probabilities:")
        for name, probability in zip(class_names, probabilities[index].tolist()):
            print(f"  {name}: {probability:.6f}")
        print("Top-4 probabilities (highest first):")
        for class_index, probability in zip(
            top_classes[index].tolist(), top_probs[index].tolist()
        ):
            print(f"  {class_names[class_index]}: {probability:.6f}")

        ax.imshow(image.squeeze(0).numpy(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"Predicted: {predicted_name}\nTrue: {true_name}")
        ax.axis("off")

    fig.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()


### 9 count_params.py

In [ ]:
# %load count_params.py
"""Print the total number of parameters in ImageClassifier."""

from model import ImageClassifier


def main():
    model = ImageClassifier()
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    print(f"Total model parameters: {parameter_count:,}")


if __name__ == "__main__":
    main()
